# MCP Client: Consuming a Custom MCP Server

**Definition:** MCP (Model Context Protocol) is a standard way for an application to expose tools and resources to an LLM, independent of which model or vendor is calling them. An **MCP server** defines the tools (like our `custom-mcp-python` server does); an **MCP client** is the piece that connects to that server, discovers what it offers, and bridges those tools into a conversation with Claude.

This is a different pattern from writing tools directly: normally you'd write the tool schema *and* the Python function yourself, side by side, and hand both to Claude. Here, the tool's implementation lives entirely in a separate server process — our client only needs to (1) connect, (2) ask "what tools do you have?", and (3) relay each call Claude makes to the server, without knowing anything about how the server implements them.

**What this notebook does:**

1. Connects to the local `custom-mcp-python` server (`./custom-mcp-python`) over Streamable HTTP.
2. Discovers its tools (`add`, `get-energy-prices`, `get-todos`) via the MCP protocol.
3. Converts the discovered tools into the schema Claude's Messages API expects.
4. Runs a Claude tool-use loop, except every tool call is dispatched to the MCP server instead of a local Python function.
5. Reads the server's `greeting` resource directly.

**Before running:** start the server in a separate terminal:

```
cd ./custom-mcp-python
pip install -r requirements.txt
python server.py
```

It should print `MCP server running at http://localhost:3000`.

In [18]:
# Setup: load environment variables and create the Claude client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

MCP_SERVER_URL = "http://localhost:3000/mcp"


## 1. Connecting to the MCP server

The MCP Python SDK's `streamable_http_client` opens the HTTP connection and gives us a pair of read/write streams; `ClientSession` speaks the MCP protocol over those streams. `session.initialize()` performs the MCP handshake (protocol version negotiation, capability exchange) — the same handshake VS Code or Claude Code performs when you register an MCP server.

Each cell below opens its own short-lived connection with `async with`, does one thing, then closes it — a fresh session per operation. `mcp_call` is a small helper that wraps that connect → do something → disconnect pattern, since we'll repeat it for every step in this notebook.


In [ ]:
from mcp.client.streamable_http import streamable_http_client
from mcp.client.session import ClientSession


async def mcp_call(fn):
    """Open a fresh MCP connection, run fn(session), then close the connection."""
    async with streamable_http_client(MCP_SERVER_URL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            return await fn(session)


async def _check_connection(session):
    return "connected"


print(await mcp_call(_check_connection))


connected


## 2. Discovering tools

`session.list_tools()` asks the server what it offers — this is the MCP equivalent of reading a tool's docstring, except it happens at runtime over the wire instead of being hardcoded in our client. Each MCP `Tool` has a `name`, `description`, and `input_schema` (JSON Schema), which map directly onto the `name`/`description`/`input_schema` fields Claude's API expects for a tool definition.


In [20]:
async def _list_tools(session):
    return await session.list_tools()


mcp_tools_result = await mcp_call(_list_tools)

for tool in mcp_tools_result.tools:
    print(f"- {tool.name}: {tool.description}")
    print(f"  {tool.input_schema}")


- add: 
  {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'type': 'object', 'title': 'addArguments'}
- get-energy-prices: 
  {'properties': {}, 'type': 'object', 'title': 'get_energy_pricesArguments'}
- get-todos: 
  {'properties': {}, 'type': 'object', 'title': 'get_todosArguments'}


## 3. Converting MCP tools into Claude tool schemas

Claude doesn't know what MCP is — it just needs a list of `{name, description, input_schema}` dicts, same as any tool-use example. We build that list from what the server told us in step 2, instead of writing it by hand.


In [21]:
def mcp_tools_to_claude_schema(mcp_tools):
    return [
        {
            "name": tool.name,
            "description": tool.description or tool.name,
            "input_schema": tool.input_schema,
        }
        for tool in mcp_tools
    ]


claude_tools = mcp_tools_to_claude_schema(mcp_tools_result.tools)
claude_tools


[{'name': 'add',
  'description': 'add',
  'input_schema': {'properties': {'a': {'title': 'A', 'type': 'number'},
    'b': {'title': 'B', 'type': 'number'}},
   'required': ['a', 'b'],
   'type': 'object',
   'title': 'addArguments'}},
 {'name': 'get-energy-prices',
  'description': 'get-energy-prices',
  'input_schema': {'properties': {},
   'type': 'object',
   'title': 'get_energy_pricesArguments'}},
 {'name': 'get-todos',
  'description': 'get-todos',
  'input_schema': {'properties': {},
   'type': 'object',
   'title': 'get_todosArguments'}}]

## 4. Running the tool-use loop against the MCP server

This follows the standard tool-use `while` loop: send messages to Claude, and if it asks for a tool, run it and send the result back, repeating until Claude has a final answer. The one difference here is `run_tool`: instead of calling local Python code directly, it opens an MCP connection and calls `session.call_tool(name, arguments)` — which sends the call over HTTP to the MCP server and returns its result. Claude never knows the tool lives on a separate server; from its side, this looks like any other tool call.

In [22]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content})


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


async def run_tool(tool_name, tool_input):
    async def _call(session):
        return await session.call_tool(tool_name, tool_input)

    result = await mcp_call(_call)
    return "\n".join(block.text for block in result.content if block.type == "text")


async def run_tools(message):
    tool_result_blocks = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            output = await run_tool(block.name, block.input)
            tool_result_blocks.append(
                {"type": "tool_result", "tool_use_id": block.id, "content": output, "is_error": False}
            )
        except Exception as e:
            tool_result_blocks.append(
                {"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True}
            )
    return tool_result_blocks


async def run_conversation(messages, tools):
    while True:
        response = client.messages.create(
            model=model, max_tokens=1000, messages=messages, tools=tools
        )
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        add_user_message(messages, await run_tools(response))

    return messages


In [23]:
messages = []
add_user_message(messages, "What is 47 plus 128? Use the add tool to be sure.")

await run_conversation(messages, tools=claude_tools)



47 plus 128 equals **175**.


[{'role': 'user',
  'content': 'What is 47 plus 128? Use the add tool to be sure.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_013wb9NLXfjt7hy3EifnCNtw', caller=DirectCaller(type='direct'), input={'a': 47, 'b': 128}, name='add', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013wb9NLXfjt7hy3EifnCNtw',
    'content': '175.0',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='47 plus 128 equals **175**.', type='text')]}]

Try a second query that uses one of the server's other tools — `get-todos` fetches a public todo list, so Claude will need to call it and then summarize the response itself.


In [24]:
messages = []
add_user_message(messages, "Get first 10 todo details")

await run_conversation(messages, tools=claude_tools)



Here are the details for the first 10 todos:

1. **ID: 1** - User 1
   - Title: "delectus aut autem"
   - Status: Not completed

2. **ID: 2** - User 1
   - Title: "quis ut nam facilis et officia qui"
   - Status: Not completed

3. **ID: 3** - User 1
   - Title: "fugiat veniam minus"
   - Status: Not completed

4. **ID: 4** - User 1
   - Title: "et porro tempora"
   - Status: **Completed** ✓

5. **ID: 5** - User 1
   - Title: "laboriosam mollitia et enim quasi adipisci quia provident illum"
   - Status: Not completed

6. **ID: 6** - User 1
   - Title: "qui ullam ratione quibusdam voluptatem quia omnis"
   - Status: Not completed

7. **ID: 7** - User 1
   - Title: "illo expedita consequatur quia in"
   - Status: Not completed

8. **ID: 8** - User 1
   - Title: "quo adipisci enim quam ut ab"
   - Status: **Completed** ✓

9. **ID: 9** - User 1
   - Title: "molestiae perspiciatis ipsa"
   - Status: Not completed

10. **ID: 10** - User 1
    - Title: "illo est ratione doloremque quia maiore

[{'role': 'user', 'content': 'Get first 10 todo details'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_014j6XWt85RrzkMXFdomfAYP', caller=DirectCaller(type='direct'), input={}, name='get-todos', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_014j6XWt85RrzkMXFdomfAYP',
    'content': "Todos: [{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}, {'userId': 1, 'id': 2, 'title': 'quis ut nam facilis et officia qui', 'completed': False}, {'userId': 1, 'id': 3, 'title': 'fugiat veniam minus', 'completed': False}, {'userId': 1, 'id': 4, 'title': 'et porro tempora', 'completed': True}, {'userId': 1, 'id': 5, 'title': 'laboriosam mollitia et enim quasi adipisci quia provident illum', 'completed': False}, {'userId': 1, 'id': 6, 'title': 'qui ullam ratione quibusdam voluptatem quia omnis', 'completed': False}, {'userId': 1, 'id': 7, 'title': 'illo expedita consequatur quia in', 'complete

## 5. Reading a resource

Resources are MCP's other primitive besides tools — addressable data, fetched by URI rather than "called" with arguments. The server's `greeting` resource is a template (`greeting://{name}`); we read it directly through the client, the same way a client would fetch any other resource URI.


In [25]:
async def _read_greeting(session):
    return await session.read_resource("greeting://World")


greeting = await mcp_call(_read_greeting)

for content in greeting.contents:
    print(content.text)


Hello, World!
